# DSC LOGIKA UI 2026 ? Feature engineering historis, tuning LightGBM, dan submission

Notebook ini membangun fitur dari tabel resmi panitia, menerapkan cutoff `decision_datetime`, mengevaluasi KS dengan Stratified 5-Fold, lalu menyimpan model dan file submission. Jalankan dari direktori proyek yang berisi folder `data-science-competition-logika-ui-2026`. Seed ditetapkan agar proses dapat direproduksi.

**Catatan cutoff:** fitur hanya memakai observasi bertanggal sampai waktu keputusan (inklusif). Riwayat pembayaran dibatasi ke angsuran yang jatuh tempo sampai cutoff; keterlambatan dihitung dari tanggal pembayaran yang sudah terjadi atau umur tunggakan pada cutoff.


In [ ]:
from pathlib import Path
import json
import gc
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from scipy.stats import ks_2samp

SEED = 42
N_FOLDS = 5
DATA_DIR = Path('data-science-competition-logika-ui-2026')
OUTPUT_DIR = Path('artifacts')
OUTPUT_DIR.mkdir(exist_ok=True)

pd.set_option('display.max_columns', 100)
warnings.filterwarnings('ignore', category=FutureWarning)


## 1. Memuat pengajuan dan profil

In [ ]:
train = pd.read_csv(DATA_DIR / 'train_applications.csv', parse_dates=['application_datetime', 'decision_datetime'])
test = pd.read_csv(DATA_DIR / 'test_applications.csv', parse_dates=['application_datetime', 'decision_datetime'])
y = train.pop('target').astype('int8')
applications = pd.concat([train, test], ignore_index=True)
applications['_row_id'] = np.arange(len(applications))
applications['_cutoff'] = applications['decision_datetime']

profiles = pd.read_csv(DATA_DIR / 'customer_profiles.csv')
base = applications.merge(profiles, on='customer_id', how='left', validate='many_to_one')
base['debt_to_income'] = base['proposed_monthly_installment'] / base['declared_monthly_income'].replace(0, np.nan)
base['loan_to_income'] = base['loan_amount'] / (12 * base['declared_monthly_income'].replace(0, np.nan))
base['installment_to_income'] = base['proposed_monthly_installment'] / base['declared_monthly_income'].replace(0, np.nan)
base['application_lead_hours'] = (base['decision_datetime'] - base['application_datetime']).dt.total_seconds() / 3600
print('Train/test:', len(train), len(test), '| positive rate:', round(y.mean(), 4))
base.head()


## 2. Utilitas agregasi dengan cutoff

Tabel besar dibaca per chunk agar penggunaan memori lebih terkendali. Setiap baris transaksi dicocokkan ke pengajuan customer terkait, lalu observasi setelah cutoff dibuang sebelum agregasi. Agregasi menghasilkan fitur per `application_id`.


In [ ]:
def chunked_customer_features(filename, date_col, value_cols, categorical_cols=(), chunksize=500_000, status_col=None, valid_status=None):
    """Agregasi per pengajuan; statistik antar-chunk digabung secara matematis."""
    usecols = ['customer_id', date_col, *value_cols, *categorical_cols]
    if status_col:
        usecols.append(status_col)
    customers = set(applications['customer_id'].unique())
    pieces = []
    for chunk in pd.read_csv(DATA_DIR / filename, usecols=usecols, parse_dates=[date_col], chunksize=chunksize):
        chunk = chunk[chunk['customer_id'].isin(customers)]
        if chunk.empty:
            continue
        if status_col and valid_status is not None:
            chunk = chunk[chunk[status_col].isin(valid_status)]
        joined = chunk.merge(base[['_row_id', 'application_id', 'customer_id', '_cutoff']], on='customer_id', how='inner')
        joined = joined[joined[date_col].notna() & (joined[date_col] <= joined['_cutoff'])]
        if joined.empty:
            continue
        joined['_days_before_cutoff'] = (joined['_cutoff'] - joined[date_col]).dt.total_seconds() / 86400
        keys = ['_row_id', 'application_id']
        grouped = joined.groupby(keys, sort=False)
        out = grouped.size().rename('_event_count').to_frame()
        for col in value_cols:
            values = pd.to_numeric(joined[col], errors='coerce')
            temp = joined[keys].copy()
            temp['_v'] = values
            stats = temp.groupby(keys, sort=False)['_v'].agg(['count','sum','min','max','std'])
            stats = stats.rename(columns={s:f'{col}_{s}' for s in stats.columns})
            # Sum of squares permits correct pooled standard deviation across chunks.
            temp['_v2'] = values * values
            stats[f'{col}_sumsq'] = temp.groupby(keys, sort=False)['_v2'].sum()
            out = out.join(stats)
        for days in (90, 180, 365):
            recent = joined[joined['_days_before_cutoff'] <= days]
            if not recent.empty:
                rg = recent.groupby(keys, sort=False)
                recent_count = rg.size().rename(f'last_{days}d_event_count')
                out = out.join(recent_count)
                for col in value_cols:
                    recent_sum = rg[col].sum(min_count=1).rename(f'last_{days}d_{col}_sum')
                    out = out.join(recent_sum)
        for col in categorical_cols:
            dummies = pd.get_dummies(joined[col].fillna('missing'), prefix=f'{filename.removesuffix(".csv")}_{col}')
            dummies[keys[0]] = joined['_row_id'].to_numpy()
            dummies[keys[1]] = joined['application_id'].to_numpy()
            out = out.join(dummies.groupby(keys, sort=False).sum())
            if filename == 'ecommerce_transactions.csv' and col in ('product_category', 'payment_method'):
                spend = joined.groupby(keys + [col], sort=False)['item_total'].sum().unstack(col, fill_value=0)
                spend.columns = [f'ecommerce_item_total_{col}_{value}' for value in spend.columns]
                out = out.join(spend)
        pieces.append(out.reset_index())
        del chunk, joined, out
        gc.collect()
    if not pieces:
        return pd.DataFrame({'_row_id': pd.Series(dtype='int64')})
    combined = pd.concat(pieces, ignore_index=True)
    keys = ['_row_id','application_id']
    result = combined.groupby(keys, as_index=False, sort=False).sum(numeric_only=True)
    for col in value_cols:
        result[f'{col}_min'] = combined.groupby(keys, sort=False)[f'{col}_min'].min().to_numpy()
        result[f'{col}_max'] = combined.groupby(keys, sort=False)[f'{col}_max'].max().to_numpy()
    for col in value_cols:
        n = result[f'{col}_count'].replace(0, np.nan)
        result[f'{filename.removesuffix(".csv")}_{col}_mean'] = result[f'{col}_sum'] / n
        result[f'{filename.removesuffix(".csv")}_{col}_sum'] = result[f'{col}_sum']
        result[f'{filename.removesuffix(".csv")}_{col}_std'] = np.sqrt(((result[f'{col}_sumsq'] - result[f'{col}_sum']**2 / n).clip(lower=0) / (n-1)).where(n > 1))
        for stat in ['min','max']:
            result[f'{filename.removesuffix(".csv")}_{col}_{stat}'] = result[f'{col}_{stat}']
        result.drop(columns=[f'{col}_{s}' for s in ['count','sum','min','max','std','sumsq']], inplace=True)
    result.rename(columns={'_event_count': f'{filename.removesuffix(".csv")}_event_count'}, inplace=True)
    return result

def merge_features(frame, features):
    if len(features) and '_row_id' in features:
        return frame.merge(features, on=['_row_id', 'application_id'], how='left', validate='one_to_one')
    return frame


## 3. Fitur rekening, transaksi, dan biro kredit

In [ ]:
# Perilaku rekening bulanan: hanya bulan yang telah selesai/tercatat sebelum cutoff.
account_features = chunked_customer_features(
    'account_behavior.csv', 'month',
    ['opening_balance','closing_balance','inflows','outflows','income_like_credits','average_balance','minimum_balance','low_balance_days']
)
base = merge_features(base, account_features)
del account_features; gc.collect()

# E-commerce: hanya pesanan completed yang terjadi sebelum cutoff. order_total berulang per item,
# sehingga tidak dipakai sebagai nominal belanja; item_total dijumlahkan per item.
ecom_features = chunked_customer_features(
    'ecommerce_transactions.csv', 'transaction_datetime', ['quantity','item_total'],
    categorical_cols=['product_category','payment_method'], status_col='order_status', valid_status=['completed']
)
base = merge_features(base, ecom_features)
del ecom_features; gc.collect()

# Biro kredit sudah mempunyai application_id; filter waktu ketersediaan informasi (available_datetime).
bureau = pd.read_csv(DATA_DIR / 'bureau_observations.csv', parse_dates=['available_datetime'])
bureau = bureau.merge(base[['_row_id','application_id','_cutoff']], on='application_id', how='inner')
bureau = bureau[bureau['available_datetime'] <= bureau['_cutoff']]
bureau_cols = ['score','active_accounts','outstanding_balance','revolving_credit_limit','revolving_outstanding_balance','utilization','inquiries_last_6m','delinquent_accounts_last_12m']
bureau_agg = bureau.groupby(['_row_id','application_id'])[bureau_cols].agg(['mean','max','min','sum','std'])
bureau_agg.columns = [f'bureau_{c}_{s}' for c,s in bureau_agg.columns]
bureau_agg = bureau_agg.join(bureau.groupby(['_row_id','application_id']).size().rename('bureau_observation_count')).reset_index()
base = merge_features(base, bureau_agg)
# Provider-level summaries preserve differences that can be hidden by an overall mean.
provider_frames = []
for provider in bureau['bureau_provider'].dropna().unique():
    bp = bureau[bureau['bureau_provider'] == provider].groupby(['_row_id','application_id'])[bureau_cols].agg(['mean','max'])
    bp.columns = [f'bureau_{provider}_{col}_{stat}' for col,stat in bp.columns]
    provider_frames.append(bp.reset_index())
for bp in provider_frames:
    base = merge_features(base, bp)
# Keep the most recent record that was actually available by decision time.
latest = bureau.sort_values('available_datetime').drop_duplicates(['_row_id','application_id'], keep='last')
latest_cols = ['score','active_accounts','outstanding_balance','revolving_outstanding_balance','utilization','inquiries_last_6m','delinquent_accounts_last_12m']
latest = latest[['_row_id','application_id'] + latest_cols].rename(columns={c:f'bureau_latest_{c}' for c in latest_cols})
base = base.merge(latest, on=['_row_id','application_id'], how='left', validate='one_to_one')
del bureau, bureau_agg, provider_frames, bp, latest; gc.collect()
print('Feature table:', base.shape)


## 4. Aplikasi sebelumnya dan riwayat pembayaran

Aplikasi masa lalu dipilih hanya jika keputusan historisnya sudah tersedia sebelum cutoff pengajuan saat ini. Pembayaran dipasangkan melalui `loan_id` dari aplikasi terdahulu dan hanya angsuran yang jatuh tempo pada atau sebelum cutoff yang dihitung.


In [ ]:
prev = pd.read_csv(DATA_DIR / 'previous_applications.csv', parse_dates=['application_datetime','decision_datetime'])
prev = prev.rename(columns={'application_id':'previous_application_id'})
prev_join = base[['_row_id','application_id','customer_id','_cutoff']].merge(prev, on='customer_id', how='left')
prev_join = prev_join[(prev_join['decision_datetime'] <= prev_join['_cutoff']) & (prev_join['previous_application_id'] != prev_join['application_id'])]
prev_join['is_approved'] = (prev_join['decision'] == 'approved').astype('int8')
prev_join['is_rejected'] = (prev_join['decision'] == 'rejected').astype('int8')
prev_join['is_cancelled'] = (prev_join['decision'] == 'cancelled').astype('int8')
prev_join['historical_loan_amount'] = prev_join['approved_amount'].fillna(prev_join['requested_amount'])
prev_agg = prev_join.groupby(['_row_id','application_id']).agg(
    previous_application_count=('previous_application_id','count'),
    previous_approved_count=('is_approved','sum'), previous_rejected_count=('is_rejected','sum'),
    previous_cancelled_count=('is_cancelled','sum'), previous_amount_mean=('historical_loan_amount','mean'),
    previous_amount_max=('historical_loan_amount','max'), previous_installment_mean=('scheduled_monthly_installment','mean'),
    previous_rate_mean=('annual_interest_rate','mean'), previous_approval_rate=('is_approved','mean'),
    previous_rejection_rate=('is_rejected','mean'), previous_latest_decision=('decision_datetime','max'),
).reset_index()
base = merge_features(base, prev_agg)
base['previous_months_since_last_decision'] = (base['_cutoff'] - base['previous_latest_decision']).dt.total_seconds() / (30.4375 * 86400)
base.drop(columns=['previous_latest_decision'], inplace=True)
base['previous_installment_to_income'] = base['previous_installment_mean'] / base['declared_monthly_income'].replace(0, np.nan)
base['previous_loan_to_income'] = base['previous_amount_mean'] / (12 * base['declared_monthly_income'].replace(0, np.nan))

# Baca repayment em chunks; join via customer and cutoff, then compute DPD as observable at cutoff.
rep_chunks = []
rep_customers = set(applications['customer_id'].unique())
for chunk in pd.read_csv(DATA_DIR / 'repayment_history.csv', parse_dates=['due_date','last_payment_datetime'], chunksize=500_000):
    chunk = chunk[chunk['customer_id'].isin(rep_customers)]
    if chunk.empty: continue
    j = chunk.merge(base[['_row_id','application_id','customer_id','_cutoff']], on='customer_id', how='inner')
    j = j[j['due_date'].notna() & (j['due_date'] <= j['_cutoff'])]
    # Pembayaran sesudah cutoff tidak diketahui pada saat keputusan.
    paid_by_cutoff = j['last_payment_datetime'].notna() & (j['last_payment_datetime'] <= j['_cutoff'])
    j['_paid_by_cutoff'] = paid_by_cutoff.astype('int8')
    j['_amount_paid_by_cutoff'] = np.where(paid_by_cutoff, j['amount_paid'], 0)
    j['_dpd_at_cutoff'] = np.where(paid_by_cutoff, (j['last_payment_datetime'] - j['due_date']).dt.days.clip(lower=0), (j['_cutoff'] - j['due_date']).dt.days.clip(lower=0))
    j['_due_amount'] = j['amount_due']
    g = j.groupby(['_row_id','application_id']).agg(
        repayment_installments=('installment_id','count'), repayment_paid_count=('_paid_by_cutoff','sum'),
        repayment_due_sum=('_due_amount','sum'), repayment_paid_sum=('_amount_paid_by_cutoff','sum'),
        repayment_dpd_mean=('_dpd_at_cutoff','mean'), repayment_dpd_max=('_dpd_at_cutoff','max'),
        repayment_dpd_30_count=('_dpd_at_cutoff',lambda x:(x>=30).sum()),
        repayment_dpd_90_count=('_dpd_at_cutoff',lambda x:(x>=90).sum()),
    ).reset_index()
    rep_chunks.append(g)
    del chunk,j,g; gc.collect()
if rep_chunks:
    rep_features = pd.concat(rep_chunks, ignore_index=True).groupby(['_row_id','application_id'],as_index=False).sum(numeric_only=True)
    base = merge_features(base, rep_features)
    del rep_features,rep_chunks; gc.collect()

base['repayment_paid_rate'] = base['repayment_paid_count'] / base['repayment_installments'].replace(0, np.nan)
base['repayment_dpd30_rate'] = base['repayment_dpd_30_count'] / base['repayment_installments'].replace(0, np.nan)
base['repayment_dpd90_rate'] = base['repayment_dpd_90_count'] / base['repayment_installments'].replace(0, np.nan)
base['repayment_amount_paid_rate'] = base['repayment_paid_sum'] / base['repayment_due_sum'].replace(0, np.nan)
base['ecommerce_spend_to_income'] = base['ecommerce_transactions_item_total_sum'] / base['declared_monthly_income'].replace(0, np.nan)
base['account_inflows_to_income'] = base['account_behavior_income_like_credits_sum'] / base['declared_monthly_income'].replace(0, np.nan)
for days in (90, 180, 365):
    months = days / 30.4375
    acct = f'account_behavior_last_{days}d_income_like_credits_sum'
    spend = f'ecommerce_transactions_last_{days}d_item_total_sum'
    if acct in base:
        base[f'account_recent_income_monthly_{days}d'] = base[acct] / months
        base[f'account_recent_income_to_declared_{days}d'] = base[acct] / (months * base['declared_monthly_income'].replace(0, np.nan))
    if spend in base:
        base[f'ecommerce_recent_spend_monthly_{days}d'] = base[spend] / months
        base[f'ecommerce_recent_spend_to_income_{days}d'] = base[spend] / (months * base['declared_monthly_income'].replace(0, np.nan))
print('Feature table after history:', base.shape)


## 5. Eksperimen model dan validasi KS resmi

Pencarian LightGBM 3-fold mencoba 8 setelan: jumlah leaves 7, 15, 31, dan 63; regularisasi; sampling; `extra_trees`; serta bobot kelas. Skor terbaik screening sekitar 0,5012. Pada validasi 5-fold, konfigurasi LightGBM terbaik mendapat KS rata-rata 0,5044.

Algoritma pembanding yang diuji adalah Extra Trees, Random Forest, dan HistGradientBoosting. Extra Trees mendapat KS sekitar 0,464?0,468, Random Forest 0,4746, dan HistGradientBoosting sekitar 0,4958 pada screening. HistGradientBoosting kemudian diuji 5-fold dan dicampur dengan LightGBM. Semua angka dihitung menggunakan `score()` dari `evaluation-code.ipynb`; angka ini adalah validasi lokal, bukan skor Kaggle.

Ensemble dengan bobot HistGradientBoosting 0,15 menghasilkan OOF KS 0,5039 (rata-rata KS per fold 0,5063). Perbaikannya kecil dan target 0,58 belum tercapai.


In [ ]:
# Gunakan evaluator resmi dari evaluation-code.ipynb.
evaluation_nb = json.loads(Path('evaluation-code.ipynb').read_text(encoding='utf-8'))
evaluation_ns = {}
for evaluation_cell in evaluation_nb['cells']:
    if evaluation_cell['cell_type'] == 'code':
        exec(compile(''.join(evaluation_cell['source']), 'evaluation-code.ipynb', 'exec'), evaluation_ns)
evaluation_score = evaluation_ns['score']

def competition_ks(y_true, y_pred, ids):
    solution = pd.DataFrame({'application_id': np.asarray(ids), 'default_probability': np.asarray(y_true)})
    submission_eval = pd.DataFrame({'application_id': np.asarray(ids), 'default_probability': np.asarray(y_pred)})
    return evaluation_score(solution, submission_eval, 'application_id')

train_ids = train['application_id'].to_numpy()
folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X, y))
# Dua konfigurasi LightGBM terbaik dari eksperimen hiperparameter dan satu HistGradientBoosting.
lgb_configs = {
    'lgb_leaf31_strong_reg': dict(num_leaves=31, min_child_samples=100, reg_lambda=8.0, colsample_bytree=0.70, extra_trees=False),
    'lgb_leaf31_extra_trees': dict(num_leaves=31, min_child_samples=80, reg_lambda=5.0, colsample_bytree=0.85, extra_trees=True),
}
lightgbm_models, lightgbm_oof = {}, {}
for name, params in lgb_configs.items():
    models, oof, fold_scores = [], np.zeros(len(y)), []
    for fold, (tr_idx, va_idx) in enumerate(folds, start=1):
        model = LGBMClassifier(objective='binary', n_estimators=3000, learning_rate=0.03,
            max_depth=-1, subsample=0.85, subsample_freq=1, random_state=SEED + fold,
            n_jobs=-1, verbosity=-1, **params)
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx], eval_set=[(X.iloc[va_idx], y.iloc[va_idx])],
                  eval_metric='auc', callbacks=[early_stopping(150, verbose=False), log_evaluation(0)])
        pred = model.predict_proba(X.iloc[va_idx])[:, 1]
        oof[va_idx] = pred
        fold_scores.append(competition_ks(y.iloc[va_idx], pred, train_ids[va_idx]))
        models.append(model)
    lightgbm_models[name], lightgbm_oof[name] = models, oof
    print(f'{name}: mean fold KS={np.mean(fold_scores):.5f}, pooled OOF KS={competition_ks(y,oof,train_ids):.5f}')

lgb_oof = np.mean(list(lightgbm_oof.values()), axis=0)
# HistGradientBoosting memberi sedikit sinyal tambahan pada OOF; gunakan rata-rata 5 fold.
hist_params = dict(max_iter=200, learning_rate=0.05, max_leaf_nodes=15, l2_regularization=5.0, early_stopping=True)
hist_models, hist_oof, hist_fold_scores = [], np.zeros(len(y)), []
for fold, (tr_idx, va_idx) in enumerate(folds, start=1):
    model = HistGradientBoostingClassifier(random_state=SEED + fold, **hist_params)
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    pred = model.predict_proba(X.iloc[va_idx])[:, 1]
    hist_oof[va_idx] = pred
    hist_fold_scores.append(competition_ks(y.iloc[va_idx], pred, train_ids[va_idx]))
    hist_models.append(model)
print(f'hist_gradient_boosting: mean fold KS={np.mean(hist_fold_scores):.5f}, pooled OOF KS={competition_ks(y,hist_oof,train_ids):.5f}')

# Bobot 0.15 memberi hasil terbaik di antara bobot campuran yang diuji pada OOF.
HIST_WEIGHT = 0.15
ensemble_oof = (1 - HIST_WEIGHT) * lgb_oof + HIST_WEIGHT * hist_oof
print(f'Ensemble official OOF KS={competition_ks(y,ensemble_oof,train_ids):.5f}')
print('Selected LightGBM configurations:', list(lgb_configs))


## 6. Model ensemble dan file submission

Model LightGBM dan HistGradientBoosting, bobot blending, serta urutan fitur disimpan bersama agar inferensi bisa direproduksi.


In [ ]:
# Simpan seluruh komponen model ensemble serta bobot blending.
ensemble_bundle = {
    'lightgbm_models': [m for name in lgb_configs for m in lightgbm_models[name]],
    'lightgbm_config_names': list(lgb_configs),
    'histgradientboosting_models': hist_models,
    'histgradientboosting_weight': HIST_WEIGHT,
    'feature_names': list(X.columns),
}
joblib.dump(ensemble_bundle, OUTPUT_DIR / 'selected_ensemble.joblib', compress=3)
p_lgb = np.mean([m.predict_proba(X_test)[:, 1] for name in lgb_configs for m in lightgbm_models[name]], axis=0)
p_hist = np.mean([m.predict_proba(X_test)[:, 1] for m in hist_models], axis=0)
test_pred = np.clip((1 - HIST_WEIGHT) * p_lgb + HIST_WEIGHT * p_hist, 0.0, 1.0)
submission = pd.DataFrame({
    'application_id': test['application_id'].to_numpy(),
    'default_probability': test_pred
})
assert submission['application_id'].equals(test['application_id'])
assert submission['default_probability'].notna().all()
assert submission['default_probability'].between(0, 1).all()
submission_path = OUTPUT_DIR / 'DSC0102_JanjiLombaTerakhirTahunIni_Submisi.csv'
submission.to_csv(submission_path, index=False)
print('Saved submission:', submission_path)
submission.head()


## Catatan reproduksibilitas dan keluaran

Jalankan semua sel secara berurutan dari direktori proyek. Fitur peristiwa dibatasi pada informasi yang tersedia sampai `decision_datetime`. Seed dan pembagian Stratified 5-Fold ditetapkan.

Submission dan model ensemble disimpan di folder `artifacts/` sebagai `DSC0102_JanjiLombaTerakhirTahunIni_Submisi.csv` dan `selected_ensemble.joblib`. File model berisi model LightGBM, model HistGradientBoosting, bobot campuran, dan urutan fitur.
